# Practical selection on Kaggle T4

Same Tesla T4 as the earlier stages. Do not switch GPUs.

Uses only `data/dev.jsonl` (20 examples). Do not load the held-out split.

Question: can we automatically obtain the accuracy recovery from evidence-preserving selection, at budget 32, with little extra compute?

One sender cache per example, five receiver conditions:
1. Full relay (reference)
2. Ordinary sender-attention headwise
3. Question-name record matcher: parse the package from B's question, keep that complete inventory line. No gold locker or record index.
4. Receiver-question probe: one forward pass of B's question and choices on A's full cache; discard new KV; rank original prompt positions
5. Oracle evidence line on the same cache (ceiling, not a method)

OBF is not used. Probe extra time is compute at the sender, not extra communication. 100 receiver evaluations. Resume is on. Download `/kaggle/working/results_stage_practical` during the run.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_practical.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_practical.py").is_file() and (cand / "data" / "dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_practical.py and data/dev.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_practical.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("dev examples", sum(1 for _ in (REPO / "data" / "dev.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Download /kaggle/working/results_stage_practical during the run. Resume is on by default.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_practical", "--split", "dev"])
print("stage_practical exit", rc)

In [ ]:
src = WORK / "results_stage_practical"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_practical" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:8000])